# Top-4 Sweeps — Methods 01, 02, 04

This notebook launches the missing method sweeps one after another for the four best configurations selected from method 03: `K08`, `K11`, `K06`, `K09`.

Results are written to the same method-local locations as when each sweep is launched separately:

- `methods/01_sequence_classification/results/sweeps/...`
- `methods/02_causal_lm_generation_parsing/results/sweeps/...`
- `methods/04_causal_lm_structured_generation/results/sweeps/...`


## Setup

In [ ]:
from pathlib import Path
import os
import signal
import subprocess
import sys
import time

import pandas as pd

TOP_CONFIG_INDEXES = [8, 11, 6, 9]


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "dataset").is_dir() and (candidate / "lora-fine-tuning").is_dir():
            return candidate
    raise RuntimeError("Could not find project root.")


PROJECT_ROOT = find_project_root()
METHODS_ROOT = PROJECT_ROOT / "lora-fine-tuning" / "methods"

SWEEPS = [
    {
        "method": "01_sequence_classification",
        "script": METHODS_ROOT / "01_sequence_classification" / "qwen3_0.6b_sequence_classification_sweep.py",
        "results_root": METHODS_ROOT / "01_sequence_classification" / "results",
        "sweep_glob": "qwen3_sequence_classification_*",
    },
    {
        "method": "02_causal_lm_generation_parsing",
        "script": METHODS_ROOT / "02_causal_lm_generation_parsing" / "qwen3_0.6b_generation_parsing_sweep.py",
        "results_root": METHODS_ROOT / "02_causal_lm_generation_parsing" / "results",
        "sweep_glob": "qwen3_clm_generation_parsing_*",
    },
    {
        "method": "04_causal_lm_structured_generation",
        "script": METHODS_ROOT / "04_causal_lm_structured_generation" / "qwen3_0.6b_structured_generation_sweep.py",
        "results_root": METHODS_ROOT / "04_causal_lm_structured_generation" / "results",
        "sweep_glob": "qwen3_clm_structured_generation_*",
    },
]

# Keep this True only for a local quick check. For the overnight A100 run, set it to False.
SMOKE_TEST = False

# Optional stable IDs. Leave as None to create timestamped sweep directories.
# If a run is interrupted, paste the printed ID here and rerun with RESUME=True.
SWEEP_IDS = {
    "01_sequence_classification": None,
    "02_causal_lm_generation_parsing": None,
    "04_causal_lm_structured_generation": None,
}

RESUME = True
COOLDOWN_BETWEEN_METHODS_SECONDS = 30
COOLDOWN_BETWEEN_CONFIGS_SECONDS = 10

print(f"Project root: {PROJECT_ROOT}")
for sweep in SWEEPS:
    print(f"{sweep['method']}: {sweep['script']}")


## Utilities

In [ ]:
def terminate_process_tree(process: subprocess.Popen, timeout: float = 30.0) -> None:
    if process.poll() is not None:
        return
    if os.name == "nt":
        process.terminate()
    else:
        try:
            os.killpg(process.pid, signal.SIGINT)
        except ProcessLookupError:
            return
        except Exception:
            process.send_signal(signal.SIGINT)
    try:
        process.wait(timeout=timeout)
        return
    except subprocess.TimeoutExpired:
        pass

    if os.name == "nt":
        process.terminate()
    else:
        try:
            os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError:
            return
        except Exception:
            process.terminate()
    try:
        process.wait(timeout=10)
        return
    except subprocess.TimeoutExpired:
        pass

    if os.name == "nt":
        process.kill()
    else:
        try:
            os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError:
            return
        except Exception:
            process.kill()
    process.wait()


def latest_sweep_dir(sweep: dict) -> Path | None:
    sweeps_root = sweep["results_root"] / "sweeps"
    sweep_dirs = sorted(
        [path for path in sweeps_root.glob(sweep["sweep_glob"]) if path.is_dir()],
        key=lambda path: path.stat().st_mtime,
    )
    return sweep_dirs[-1] if sweep_dirs else None


def print_failure_summary(sweep: dict) -> None:
    sweep_dir = latest_sweep_dir(sweep)
    if sweep_dir is None:
        print(f"No sweep directory found for {sweep['method']}.")
        return
    summary_path = sweep_dir / "summary.csv"
    if not summary_path.exists():
        print(f"No summary.csv found in {sweep_dir}")
        return
    summary = pd.read_csv(summary_path)
    failed = summary[summary["status"].isin(["failed", "interrupted", "metrics_json_invalid"])]
    if failed.empty:
        print(f"No failed rows in {summary_path}")
        return
    display(failed[["config_index", "status", "error_type", "error_message", "run_dir"]])


def run_command(command: list[str], sweep: dict) -> None:
    print(" ".join(command), flush=True)
    process = subprocess.Popen(
        command,
        cwd=PROJECT_ROOT,
        text=True,
        start_new_session=(os.name != "nt"),
    )
    try:
        return_code = process.wait()
    except KeyboardInterrupt:
        terminate_process_tree(process)
        raise
    if return_code != 0:
        print_failure_summary(sweep)
        raise RuntimeError(f"{sweep['method']} exited with code {return_code}; see failed config details above.")


## Preview Selected Configurations

In [ ]:
for sweep in SWEEPS:
    print(f"\n=== {sweep['method']} ===")
    subprocess.run(
        [sys.executable, str(sweep["script"]), "list-configs"],
        cwd=PROJECT_ROOT,
        check=True,
    )


## Launch Overnight Run

In [ ]:
started_at = time.time()
completed_methods = []

for offset, sweep in enumerate(SWEEPS):
    method = sweep["method"]
    print(f"\n==============================")
    print(f"Starting {method}")
    print(f"Results root: {sweep['results_root']}")
    print(f"Selected configs: {TOP_CONFIG_INDEXES}")
    print(f"==============================", flush=True)

    command = [
        sys.executable,
        str(sweep["script"]),
        "run-sweep",
        "--results-root",
        str(sweep["results_root"]),
        "--cooldown-seconds",
        str(COOLDOWN_BETWEEN_CONFIGS_SECONDS),
    ]
    if RESUME:
        command.append("--resume")
    sweep_id = SWEEP_IDS.get(method)
    if sweep_id:
        command.extend(["--sweep-id", sweep_id])

    # The scripts already default to K08, K11, K06, K09, but pass them explicitly
    # to make the overnight run self-documenting in the notebook output.
    command.extend(["--config-index", ",".join(str(index) for index in TOP_CONFIG_INDEXES)])

    if SMOKE_TEST:
        command.extend([
            "--allow-non-cuda",
            "--max-steps", "1",
            "--train-limit", "24",
            "--validation-limit", "8",
            "--test-limit", "8",
            "--cooldown-seconds", "0",
        ])
    else:
        cuda_check = subprocess.run(
            [sys.executable, "-c", "import torch; raise SystemExit(0 if torch.cuda.is_available() else 1)"],
            cwd=PROJECT_ROOT,
        )
        if cuda_check.returncode != 0:
            raise RuntimeError("CUDA is not available. Set SMOKE_TEST=True for local checks or run this on the A100 box.")

    run_command(command, sweep)
    completed_methods.append(method)

    if offset < len(SWEEPS) - 1 and COOLDOWN_BETWEEN_METHODS_SECONDS > 0:
        print(f"Cooling down between methods for {COOLDOWN_BETWEEN_METHODS_SECONDS:g} seconds.")
        time.sleep(COOLDOWN_BETWEEN_METHODS_SECONDS)

elapsed = time.time() - started_at
print(f"Completed methods: {completed_methods}")
print(f"Elapsed seconds: {elapsed:.1f}")


## Summaries

In [ ]:
summaries = []
for sweep in SWEEPS:
    sweep_dir = latest_sweep_dir(sweep)
    if sweep_dir is None:
        print(f"No sweep found for {sweep['method']}")
        continue
    summary_path = sweep_dir / "summary.csv"
    if not summary_path.exists():
        print(f"No summary.csv for {sweep['method']} at {summary_path}")
        continue
    summary = pd.read_csv(summary_path)
    summary.insert(0, "method", sweep["method"])
    summaries.append(summary)
    print(f"{sweep['method']}: {summary_path}")

if summaries:
    combined = pd.concat(summaries, ignore_index=True, sort=False)
    display(combined.sort_values(["method", "status", "validation_f1", "test_f1"], ascending=[True, True, False, False]))
else:
    print("No summaries loaded yet.")


## Rebuild Summaries

In [ ]:
for sweep in SWEEPS:
    sweep_dir = latest_sweep_dir(sweep)
    if sweep_dir is None:
        print(f"No sweep found for {sweep['method']}")
        continue
    subprocess.run(
        [sys.executable, str(sweep["script"]), "summarize", "--sweep-dir", str(sweep_dir)],
        cwd=PROJECT_ROOT,
        check=True,
    )
    print(f"Rebuilt: {sweep_dir / 'summary.csv'}")
